# E10: Hyperparameter Tuning (Models A, B, C)

**Spec sections**: 5.5, 7.1, 7.2, 7.3, 7.4, 7.5.  
**Data**: DEV GOOD+BAD only; R-folds (15) and T-folds (T1–T3) from E3, joined on app_id.  
**Features**: Model A = `feature_lists/model_a.json` (WoE via refitted OptBinning); Models B and C = `feature_lists/l4_pool.json` (raw values, special codes -> NaN + indicator).  
**Outputs**: `reports/04_tuning.xlsx`, `params/chosen_params.json`, `reports/run_summaries/s04_tune.json`, MLflow run.

Stage 1: Optuna TPE on R-folds (C: 60 LightGBM trials, B: 30 EBM trials); Model A reference CV only.  
Stage 2: shortlist (mean >= best - std(best), top 5) -> T-folds -> highest t_auc_mean, then simplest within tolerance.  
Stage 3 (C): lgb.cv on R-fold repeat 1 -> best_iter -> final rounds = round(best_iter x 1.1).

In [0]:
%pip install "optbinning" "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14" "lightgbm" "optuna" "interpret-core" --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, math, time, pickle, shutil, tempfile, warnings, importlib
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import optuna
import interpret
from interpret.glassbox import ExplainableBoostingClassifier
import optbinning
from optbinning import BinningProcess

warnings.filterwarnings("ignore", category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)
TUNE = cfg["tuning"]
N_THREADS = cfg["runtime"]["lgb_num_threads"]
L1_BIN = cfg["funnel"]["L1"]["binning"]

ARTIFACTS  = common.artifacts_path(cfg)
REPORTS    = os.path.join(ARTIFACTS, "reports")
RUN_SUMS   = os.path.join(REPORTS, "run_summaries")
FOLDS_DIR  = os.path.join(ARTIFACTS, "folds")
FEAT_LISTS = os.path.join(ARTIFACTS, "feature_lists")
BIN_DIR    = os.path.join(ARTIFACTS, "binning", "L1")
PARAMS_DIR = os.path.join(ARTIFACTS, "params")
for d in (REPORTS, RUN_SUMS, PARAMS_DIR):
    os.makedirs(d, exist_ok=True)

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.acard_master"
id_col, date_col = cfg["columns"]["id"], cfg["columns"]["date"]
target_col, status_col = cfg["columns"]["target"], cfg["columns"]["status"]

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
VERSIONS = {"sklearn": sklearn.__version__, "lightgbm": lgb.__version__, "optuna": optuna.__version__,
            "interpret": interpret.__version__, "optbinning": optbinning.__version__,
            "numpy": np.__version__, "pandas": pd.__version__}

def rnd(x):
    """Round half up (avoid Python banker's rounding)."""
    return int(math.floor(x + 0.5))

_TMP = tempfile.mkdtemp(prefix="e10_")
_t0 = time.time()
print(f"Project root: {PROJECT_ROOT}")
print(f"Git commit  : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]}")
print(f"Versions    : {VERSIONS}")
print(f"Tuning cfg  : {TUNE} | lgb_num_threads={N_THREADS}")

In [0]:
# ── DEV only: Spark filter then common.load_window (OOT never read) ──
master_dev = spark.table(MASTER_TBL).filter("window = 'DEV'").toPandas()
assert set(master_dev["window"].unique()) == {"DEV"}, "non-DEV rows loaded"
oot_s, oot_e = cfg["windows"]["OOT"]
_d_all = pd.to_datetime(master_dev[date_col])
N_OOT_ROWS_READ = int(((_d_all >= oot_s) & (_d_all < oot_e)).sum())
assert N_OOT_ROWS_READ == 0, "OOT-dated rows present in the loaded data"

dev = common.load_window(master_dev, "DEV", cfg)
dev_gb = dev[dev[status_col].isin(cfg["labelled_statuses"])]
# Keep the DEV population frozen at E3 (folds/r_folds.parquet); later appends to the master table
# (FWD/STRESS rebuild) added DEV records that the funnel and folds were never built on
frozen_ids = set(pd.read_parquet(os.path.join(FOLDS_DIR, "r_folds.parquet"), columns=["app_id"])["app_id"].astype(str))
n_new = int((~dev_gb[id_col].astype(str).isin(frozen_ids)).sum())
print(f"DEV GOOD+BAD records not in E3 folds (excluded): {n_new}")
dev_gb = dev_gb[dev_gb[id_col].astype(str).isin(frozen_ids)].reset_index(drop=True)
common.assert_labelled(dev_gb, cfg)
y = (dev_gb[status_col] == "BAD").astype(int)
exp = cfg["expected_counts"]["DEV"]
print(f"Actual dev_gb rows: {len(dev_gb)}, Bad labels: {int(y.sum())}")
assert len(dev_gb) == exp["records"] and int(y.sum()) == exp["bads"], "DEV counts differ from expected_counts"
print(f"DEV GOOD+BAD: {len(dev_gb):,} rows, {int(y.sum())} Bads (expected {exp['records']:,}/{exp['bads']})")

dev_gb["app_id"] = dev_gb[id_col].astype(str)
assert dev_gb["app_id"].is_unique, "app_id not unique in DEV GOOD+BAD — fold join would be ambiguous"
pos = pd.Series(np.arange(len(dev_gb)), index=dev_gb["app_id"])

def build_folds(fdf, keys):
    out = []
    for key, g in fdf.groupby(keys, sort=True):
        if isinstance(key, tuple) and len(key) == 1:
            key = key[0]
        tr = pos.reindex(g.loc[g["part"] == "train", "app_id"].astype(str))
        va = pos.reindex(g.loc[g["part"] == "valid", "app_id"].astype(str))
        assert tr.notna().all() and va.notna().all(), f"fold {key}: app_id not in DEV GOOD+BAD"
        tr, va = tr.to_numpy(int), va.to_numpy(int)
        assert len(np.intersect1d(tr, va)) == 0, f"fold {key}: train/valid overlap"
        out.append((key, np.sort(tr), np.sort(va)))
    return out

rf = pd.read_parquet(os.path.join(FOLDS_DIR, "r_folds.parquet"))
tf = pd.read_parquet(os.path.join(FOLDS_DIR, "t_folds.parquet"))
assert set(rf["app_id"].astype(str)) == set(dev_gb["app_id"]), "R-fold ids != DEV GOOD+BAD ids"
assert set(tf["app_id"].astype(str)) <= set(dev_gb["app_id"]), "T-fold ids outside DEV GOOD+BAD"
R = build_folds(rf, ["repeat", "fold"])
T = build_folds(tf, ["fold"])
for key, tr, va in R:
    assert len(tr) + len(va) == len(dev_gb), f"R-fold {key} does not cover DEV"
R_IDX = [(tr, va) for _, tr, va in R]
R1_IDX = [(tr, va) for k, tr, va in R if k[0] == 0]
R_TRAIN_SIZE = float(np.mean([len(tr) for tr, _ in R_IDX]))
exp_r = cfg["cv"]["r_folds"]["n_splits"] * cfg["cv"]["r_folds"]["n_repeats"]
assert len(R_IDX) == exp_r and len(R1_IDX) == cfg["cv"]["r_folds"]["n_splits"]
assert [k for k, _, _ in T] == sorted(cfg["cv"]["t_folds"].keys())
print(f"R-folds: {len(R_IDX)} (repeat-1: {len(R1_IDX)}), mean train size {R_TRAIN_SIZE:.1f}")
for k, tr, va in T:
    print(f"  {k}: train {len(tr):,} ({int(y.iloc[tr].sum())} Bads)  valid {len(va):,} ({int(y.iloc[va].sum())} Bads)")

with open(os.path.join(FEAT_LISTS, "model_a.json")) as f:
    model_a_list = json.load(f)
A_FEATS = [r["feature"] for r in model_a_list["features"]]
with open(os.path.join(FEAT_LISTS, "l4_pool.json")) as f:
    POOL = [r["feature"] for r in json.load(f)]
assert set(A_FEATS) <= set(POOL), "Model A features not a subset of the L4 pool"
fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
fdx = fdict.set_index("feature")
print(f"Model A features: {len(A_FEATS)} | L4 pool (B, C): {len(POOL)}")

In [0]:
# ── Raw matrix for B and C (special codes -> NaN + indicator) and monotone vector for C ──
binning_objects = {}
for f in POOL:
    with open(os.path.join(BIN_DIR, f"{f}.pkl"), "rb") as fh:
        binning_objects[f] = pickle.load(fh)

X_raw = common.prepare_raw(dev_gb, POOL, fdict)
COLS = list(X_raw.columns)
CAT_COLS = [f for f in POOL if str(fdx.loc[f, "dtype"]).lower() == "categorical"]
IND_COLS = [c for c in COLS if c not in POOL]
MONO = common.monotone_vector(COLS, POOL, binning_objects, fdict)
assert len(MONO) == len(COLS)

# Trend table: derived L1 bin trend vs feature-log label
flog = spark.table(f"{cat}.{sch}.acard_feature_log").toPandas().set_index("feature")
trend_df = pd.DataFrame({
    "column": COLS,
    "is_indicator": [c in IND_COLS for c in COLS],
    "l1_bin_trend": [common.binning_trend(binning_objects[c]) if c in POOL else "indicator" for c in COLS],
    "feature_log_trend": [flog.loc[c, "trend"] if c in flog.index and "trend" in flog.columns else None for c in COLS],
    "expected_direction": [fdx.loc[c, "expected_direction"] if c in fdx.index else None for c in COLS],
    "monotone": MONO,
})
TREND_NOTE = ("Monotone constraints use the trend of the fitted L1 bins (regular-bin Bad Rates, spec 7.5). "
              "The feature-log 'trend' labels (non_monotonic / peak / valley) do not match the fitted bins, "
              "which are all monotonic under auto_asc_desc.")
n_mismatch = int((trend_df.loc[~trend_df.is_indicator, "feature_log_trend"].astype(str)
                  != trend_df.loc[~trend_df.is_indicator, "l1_bin_trend"].astype(str)).sum())
print(f"Raw matrix: {X_raw.shape} | categorical: {CAT_COLS} | indicators: {IND_COLS}")
print(f"Monotone: +1={MONO.count(1)}, -1={MONO.count(-1)}, 0={MONO.count(0)} | feature-log trend label mismatches: {n_mismatch}")

# Record the trend-label discrepancy once (idempotent)
oq_path = os.path.join(REPORTS, "open_questions.md")
marker = "[E10-trend-label]"
existing = open(oq_path).read() if os.path.exists(oq_path) else ""
if n_mismatch and marker not in existing:
    with open(oq_path, "w") as fh:   # UC volumes do not support append mode: rewrite whole file
        fh.write(existing + f"\n- {marker} E10 ({datetime.now():%Y-%m-%d}): {n_mismatch} pool features have a feature-log `trend` "
                 f"label that differs from the fitted L1 bins (e.g. labelled peak/valley/non_monotonic while the bins "
                 f"are monotonic). E10 derives LightGBM monotone constraints from the fitted bins per spec 7.5. "
                 f"Model owner to confirm, and E5 `trend` column to be reviewed.\n")
    print("Note written to open_questions.md")
display(trend_df)

In [0]:
# ── Model A: Pipeline(BinningProcess(L1 params, special codes), unpenalised LR) on 15 R-folds; no tuning ──
A_SPECIAL = {f: common.parse_special_codes(fdx.loc[f, "special_codes"]) for f in A_FEATS}
A_CAT = [f for f in A_FEATS if str(fdx.loc[f, "dtype"]).lower() == "categorical"]
# cat_cutoff applies to categorical features only (as in the L1 objects)
A_FIT_PARAMS = {f: {**{k: v for k, v in L1_BIN.items() if k != "cat_cutoff" or f in A_CAT},
                    "special_codes": (A_SPECIAL[f] or None)} for f in A_FEATS}

# Cross-check against the stored L1 binning objects
param_check = []
for f in A_FEATS:
    gp = binning_objects[f].get_params()
    for k, v in A_FIT_PARAMS[f].items():
        param_check.append({"feature": f, "param": k, "config": str(v), "l1_object": str(gp.get(k)),
                            "match": str(v) == str(gp.get(k))})
param_check = pd.DataFrame(param_check)
print(f"Binning params matching the L1 objects: {param_check['match'].mean():.0%}")
if not param_check["match"].all():
    display(param_check[~param_check["match"]])

def make_pipeline_a():
    bp = BinningProcess(variable_names=A_FEATS, categorical_variables=A_CAT or None,
                        binning_fit_params=A_FIT_PARAMS)
    return Pipeline([("binning", bp), ("lr", common.lr_unpenalised(max_iter=2000))])

X_a = dev_gb[A_FEATS].copy()
for f in A_FEATS:
    if f not in A_CAT:
        X_a[f] = pd.to_numeric(X_a[f], errors="coerce")

a_rows = []
a_coef_rows = []
for (key, tr, va) in R:
    pipe = make_pipeline_a().fit(X_a.iloc[tr], y.iloc[tr])
    auc = roc_auc_score(y.iloc[va], pipe.predict_proba(X_a.iloc[va])[:, 1])
    a_rows.append({"repeat": key[0], "fold": key[1], "auc": auc})
    lr = pipe.named_steps["lr"]
    coefs = dict(zip(A_FEATS, lr.coef_[0]))
    coefs["intercept"] = float(lr.intercept_[0])
    coefs["repeat"] = key[0]
    coefs["fold"] = key[1]
    coefs["auc"] = auc
    a_coef_rows.append(coefs)
a_cv = pd.DataFrame(a_rows)
a_coefs = pd.DataFrame(a_coef_rows)
# Reorder columns: repeat, fold, auc, intercept, then features
a_coefs = a_coefs[["repeat", "fold", "auc", "intercept"] + A_FEATS]
A_AUC_MEAN, A_AUC_STD = float(a_cv["auc"].mean()), float(a_cv["auc"].std(ddof=0))
print(f"Model A 15-fold AUC: mean {A_AUC_MEAN:.4f}, std {A_AUC_STD:.4f}")

In [0]:
# ── Stage 1, Model C: Optuna TPE over the spec 7.5 ranges, lgb.cv on the 15 R-folds ──
# Fixed values from spec 7.5 (not in config.tuning). force_row_wise / feature_pre_filter / verbose are
# implementation settings for determinism and for reusing one Dataset across trials.
FIXED_C = {"objective": "binary", "metric": "auc", "bagging_freq": 1,
           "monotone_constraints": MONO, "monotone_constraints_method": "advanced",
           "deterministic": True, "seed": SEED, "num_threads": N_THREADS,
           "min_data_per_group": 100, "cat_smooth": 20,
           "force_row_wise": True, "feature_pre_filter": False, "verbose": -1}
SPACE_C = {  # spec 7.5
    "learning_rate": (0.01, 0.05, "log"), "num_leaves": (3, 15, "int"), "max_depth": (2, 4, "int"),
    "min_child_samples": (50, 200, "int"), "lambda_l2": (1.0, 30.0, "log"), "lambda_l1": (0.0, 5.0, "float"),
    "feature_fraction": (0.6, 0.9, "float"), "bagging_fraction": (0.6, 0.9, "float"),
    "min_gain_to_split": (0.0, 0.05, "float"),
}
C_PARAM_NAMES = list(SPACE_C)

def make_dataset(idx=None):
    X = X_raw if idx is None else X_raw.iloc[idx]
    yy = y if idx is None else y.iloc[idx]
    return lgb.Dataset(X, label=yy, categorical_feature=CAT_COLS, free_raw_data=False,
                       params={"feature_pre_filter": False, "verbose": -1})

def suggest_c(trial):
    p = {}
    for k, (lo, hi, kind) in SPACE_C.items():
        if kind == "int":
            p[k] = trial.suggest_int(k, lo, hi)
        else:
            p[k] = trial.suggest_float(k, lo, hi, log=(kind == "log"))
    return p

def lgb_cv(params, folds):
    res = lgb.cv({**FIXED_C, **params}, make_dataset(), num_boost_round=TUNE["lgb_max_rounds"], folds=folds,
                 callbacks=[lgb.early_stopping(TUNE["lgb_early_stopping"], verbose=False)], seed=SEED)
    k_mean = [k for k in res if k.endswith("auc-mean")][0]
    curve = np.asarray(res[k_mean]); sd = np.asarray(res[k_mean.replace("-mean", "-stdv")])
    bi = int(np.argmax(curve)) + 1
    return float(curve[bi - 1]), float(sd[bi - 1]), bi

def objective_c(trial):
    p = suggest_c(trial)
    t = time.time()
    m, s, bi = lgb_cv(p, R_IDX)
    trial.set_user_attr("cv_auc_std", s); trial.set_user_attr("best_iter", bi)
    trial.set_user_attr("seconds", round(time.time() - t, 1))
    return m

t0 = time.time()
study_c = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_c.optimize(objective_c, n_trials=TUNE["lightgbm_trials"])
c_trials = pd.DataFrame([{"trial": t.number, **t.params, "cv_auc_mean": t.value,
                          "cv_auc_std": t.user_attrs["cv_auc_std"], "best_iter": t.user_attrs["best_iter"],
                          "seconds": t.user_attrs["seconds"]} for t in study_c.trials])
print(f"Model C: {len(c_trials)} trials in {time.time() - t0:.0f}s | best mean AUC {c_trials.cv_auc_mean.max():.4f}")
display(c_trials.sort_values("cv_auc_mean", ascending=False).head(10))

In [0]:
# ── Stage 1, Model B: Optuna TPE over the spec 7.4 grid (categorical choices), 15 R-folds ──
FIXED_B = {"interactions": 0, "outer_bags": 14, "validation_size": 0.15,
           "early_stopping_rounds": 100, "random_state": SEED}           # spec 7.4
SPACE_B = {"learning_rate": [0.005, 0.01, 0.02], "max_bins": [32, 64, 128], "min_samples_leaf": [20, 50, 100],
           "max_leaves": [2, 3], "smoothing_rounds": [0, 100, 300]}       # spec 7.4
B_PARAM_NAMES = list(SPACE_B)
warnings.filterwarnings("ignore", message="Missing values detected")   # EBM plotting notice; NaN handled natively

def fit_ebm(params, idx):
    m = ExplainableBoostingClassifier(**FIXED_B, **params)
    return m.fit(X_raw.iloc[idx], y.iloc[idx])

def ebm_auc(params, tr, va):
    m = fit_ebm(params, tr)
    return roc_auc_score(y.iloc[va], m.predict_proba(X_raw.iloc[va])[:, 1])

_ebm_cache = {}   # identical parameter sets give identical results (fixed random_state)
def objective_b(trial):
    p = {k: trial.suggest_categorical(k, v) for k, v in SPACE_B.items()}
    key = tuple(p[k] for k in B_PARAM_NAMES)
    t = time.time()
    if key not in _ebm_cache:
        aucs = [ebm_auc(p, tr, va) for tr, va in R_IDX]
        _ebm_cache[key] = (float(np.mean(aucs)), float(np.std(aucs)), aucs)
        if not _ebm_cache.get("_timed"):
            _ebm_cache["_timed"] = True
            print(f"  first EBM trial: {time.time() - t:.0f}s (x{TUNE['ebm_trials']} trials upper bound)")
    m, s, aucs = _ebm_cache[key]
    trial.set_user_attr("cv_auc_std", s); trial.set_user_attr("cached", time.time() - t < 1)
    trial.set_user_attr("seconds", round(time.time() - t, 1))
    return m

t0 = time.time()
study_b = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_b.optimize(objective_b, n_trials=TUNE["ebm_trials"])
b_trials = pd.DataFrame([{"trial": t.number, **t.params, "cv_auc_mean": t.value,
                          "cv_auc_std": t.user_attrs["cv_auc_std"], "cached": t.user_attrs["cached"],
                          "seconds": t.user_attrs["seconds"]} for t in study_b.trials])
print(f"Model B: {len(b_trials)} trials ({b_trials[B_PARAM_NAMES].drop_duplicates().shape[0]} distinct) "
      f"in {time.time() - t0:.0f}s | best mean AUC {b_trials.cv_auc_mean.max():.4f}")
display(b_trials.sort_values("cv_auc_mean", ascending=False).head(10))

In [0]:
# ── Stage 2: shortlist on R-folds, evaluate on T1–T3, pick highest t_auc_mean then simplest ──
# Shortlist counts distinct parameter sets (TPE can repeat a categorical EBM set; lowest trial kept).
importlib.reload(common)
def params_of(row, names):
    out = {}
    for k in names:
        v = row[k]
        out[k] = int(v) if isinstance(v, (np.integer,)) or (k in ("num_leaves", "max_depth", "min_child_samples",
                                                                 "max_bins", "min_samples_leaf", "max_leaves",
                                                                 "smoothing_rounds")) else float(v)
    return out

def tfold_c(row):
    p = params_of(row, C_PARAM_NAMES)
    out = []
    for name, tr, va in T:
        rounds = max(1, rnd(row["best_iter"] * len(tr) / R_TRAIN_SIZE))
        bst = lgb.train({**FIXED_C, **p}, make_dataset(tr), num_boost_round=rounds)
        out.append({"tfold": name, "rounds": rounds, "auc": roc_auc_score(y.iloc[va], bst.predict(X_raw.iloc[va]))})
    return out

def tfold_b(row):
    p = params_of(row, B_PARAM_NAMES)
    return [{"tfold": name, "auc": ebm_auc(p, tr, va)} for name, tr, va in T]

def stage2(trials, model, names, tfold_fn):
    sl, thr = common.shortlist_trials(trials, cfg, param_cols=names)
    long_rows, res_rows = [], []
    for _, row in sl.iterrows():
        rs = tfold_fn(row)
        for r in rs:
            long_rows.append({"trial": int(row["trial"]), **r})
        a = np.array([r["auc"] for r in rs])
        res_rows.append({"trial": int(row["trial"]), **{k: row[k] for k in names},
                         "cv_auc_mean": row["cv_auc_mean"], "cv_auc_std": row["cv_auc_std"],
                         **{f"auc_{r['tfold']}": r["auc"] for r in rs},
                         "t_auc_mean": float(a.mean()), "t_auc_std": float(a.std(ddof=0))})
    tres = pd.DataFrame(res_rows)
    chosen, near = common.pick_simplest(tres, model, cfg)
    return sl, thr, pd.DataFrame(long_rows), tres, near, chosen

c_sl, C_THR, c_tlong, c_tres, c_near, C_CHOSEN = stage2(c_trials, "C", C_PARAM_NAMES, tfold_c)
print(f"C: shortlist threshold {C_THR:.4f} -> {len(c_sl)} trials {c_sl.trial.tolist()} | "
      f"within tolerance {c_near.trial.tolist()} | chosen trial {C_CHOSEN}")
display(c_tres)
b_sl, B_THR, b_tlong, b_tres, b_near, B_CHOSEN = stage2(b_trials, "B", B_PARAM_NAMES, tfold_b)
print(f"B: shortlist threshold {B_THR:.4f} -> {len(b_sl)} trials {b_sl.trial.tolist()} | "
      f"within tolerance {b_near.trial.tolist()} | chosen trial {B_CHOSEN}")
display(b_tres)

In [0]:
# ── Stage 3 (C): lgb.cv on R-fold repeat 1 with the chosen params -> best_iter -> final rounds ──
c_row = c_trials.set_index("trial").loc[C_CHOSEN]
C_PARAMS = params_of(c_row, C_PARAM_NAMES)
S3_AUC, S3_STD, S3_BEST_ITER = lgb_cv(C_PARAMS, R1_IDX)
C_FINAL_ROUNDS = rnd(S3_BEST_ITER * TUNE["lgb_round_multiplier"])
b_row = b_trials.set_index("trial").loc[B_CHOSEN]
B_PARAMS = params_of(b_row, B_PARAM_NAMES)
print(f"C stage 3: best_iter {S3_BEST_ITER} (AUC {S3_AUC:.4f} ± {S3_STD:.4f}) -> final rounds {C_FINAL_ROUNDS}")
print(f"C params: {C_PARAMS}")
print(f"B params: {B_PARAMS}")

In [0]:
# ── Outputs ──
def _reason(model, tres, near, chosen, keys):
    best_t = tres.loc[tres.t_auc_mean.idxmax()]
    return (f"Model {model}: highest t_auc_mean = trial {int(best_t.trial)} ({best_t.t_auc_mean:.4f}); "
            f"{len(near)} trial(s) within {TUNE['simplicity_tolerance']} ({near.trial.tolist()}); simplest by "
            f"{keys} -> trial {chosen}.")
C_REASON = _reason("C", c_tres, c_near, C_CHOSEN, [k for k, _ in common.SIMPLICITY_KEYS["C"]])
B_REASON = _reason("B", b_tres, b_near, B_CHOSEN, [k for k, _ in common.SIMPLICITY_KEYS["B"]])

meta = {"module": "E10_tune", "run_time": datetime.now().isoformat(), "git_commit": GIT_COMMIT,
        "config_sha256": CONFIG_HASH, "versions": VERSIONS}
chosen = {
    **meta,
    "A": {"features": A_FEATS, "binning_params": L1_BIN, "special_codes": A_SPECIAL,
          "categorical_features": A_CAT, "estimator": "LogisticRegression unpenalised (max_iter=2000)",
          "cv_auc_mean": A_AUC_MEAN, "cv_auc_std": A_AUC_STD},
    "B": {"input_features": POOL, "columns": COLS, "categorical_features": CAT_COLS, "trial": B_CHOSEN,
          "params": {**FIXED_B, **B_PARAMS}, "cv_auc_mean": float(b_row.cv_auc_mean),
          "cv_auc_std": float(b_row.cv_auc_std),
          "t_auc_mean": float(b_tres.set_index("trial").loc[B_CHOSEN, "t_auc_mean"]), "reason": B_REASON},
    "C": {"input_features": POOL, "columns": COLS, "categorical_feature": CAT_COLS, "trial": C_CHOSEN,
          "params": {**{k: v for k, v in FIXED_C.items() if k != "monotone_constraints"}, **C_PARAMS},
          "monotone_constraints": MONO, "num_threads": N_THREADS,
          "stage1_best_iter": int(c_row.best_iter), "stage3_best_iter": S3_BEST_ITER,
          "final_rounds": C_FINAL_ROUNDS, "round_multiplier": TUNE["lgb_round_multiplier"],
          "cv_auc_mean": float(c_row.cv_auc_mean), "cv_auc_std": float(c_row.cv_auc_std),
          "t_auc_mean": float(c_tres.set_index("trial").loc[C_CHOSEN, "t_auc_mean"]), "reason": C_REASON},
    "notes": {"trend": TREND_NOTE, "std": "cv_auc_std and t_auc_std are population std (ddof=0), as lgb.cv"},
}
CHOSEN_PATH = os.path.join(PARAMS_DIR, "chosen_params.json")
with open(CHOSEN_PATH, "w") as fh:
    json.dump(chosen, fh, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print(f"Saved {CHOSEN_PATH}")

choice_df = pd.DataFrame([
    {"model": "A", "trial": None, "cv_auc_mean": A_AUC_MEAN, "t_auc_mean": None, "params": json.dumps(L1_BIN),
     "final_rounds": None, "reason": "No tuning; reference CV with binning refitted per fold (spec 7.2, 7.3)."},
    {"model": "B", "trial": B_CHOSEN, "cv_auc_mean": float(b_row.cv_auc_mean),
     "t_auc_mean": chosen["B"]["t_auc_mean"], "params": json.dumps(B_PARAMS), "final_rounds": None, "reason": B_REASON},
    {"model": "C", "trial": C_CHOSEN, "cv_auc_mean": float(c_row.cv_auc_mean),
     "t_auc_mean": chosen["C"]["t_auc_mean"], "params": json.dumps(C_PARAMS), "final_rounds": C_FINAL_ROUNDS,
     "reason": C_REASON + f" Stage 3 best_iter {S3_BEST_ITER} x {TUNE['lgb_round_multiplier']}."},
])
fixed_df = pd.DataFrame([{"model": "C", "param": k, "value": str(v)} for k, v in FIXED_C.items()] +
                        [{"model": "B", "param": k, "value": str(v)} for k, v in FIXED_B.items()])

XLSX = os.path.join(REPORTS, "04_tuning.xlsx")
tmp = os.path.join(_TMP, "04_tuning.xlsx")
with pd.ExcelWriter(tmp, engine="openpyxl") as w:
    pd.DataFrame([{**{k: v for k, v in meta.items() if k != "versions"}, **VERSIONS}]).to_excel(w, sheet_name="meta", index=False)
    choice_df.to_excel(w, sheet_name="choice", index=False)
    a_cv.to_excel(w, sheet_name="A_cv", index=False)
    a_coefs.to_excel(w, sheet_name="A_coefs", index=False)
    param_check.to_excel(w, sheet_name="A_binning_params", index=False)
    c_trials.to_excel(w, sheet_name="C_trials", index=False)
    b_trials.to_excel(w, sheet_name="B_trials", index=False)
    c_sl.to_excel(w, sheet_name="C_shortlist", index=False)
    b_sl.to_excel(w, sheet_name="B_shortlist", index=False)
    c_tres.to_excel(w, sheet_name="C_tfold", index=False)
    b_tres.to_excel(w, sheet_name="B_tfold", index=False)
    c_tlong.to_excel(w, sheet_name="C_tfold_detail", index=False)
    b_tlong.to_excel(w, sheet_name="B_tfold_detail", index=False)
    c_near.to_excel(w, sheet_name="C_within_tolerance", index=False)
    b_near.to_excel(w, sheet_name="B_within_tolerance", index=False)
    fixed_df.to_excel(w, sheet_name="fixed_params", index=False)
    trend_df.to_excel(w, sheet_name="monotone", index=False)
    pd.DataFrame({"note": [TREND_NOTE, chosen["notes"]["std"],
                           f"Shortlist thresholds: C {C_THR:.5f}, B {B_THR:.5f}",
                           f"T-fold LightGBM rounds = stage-1 best_iter x T-train / {R_TRAIN_SIZE:.1f}, no early stopping"]}
                 ).to_excel(w, sheet_name="notes", index=False)
shutil.copyfile(tmp, XLSX)
print(f"Saved {XLSX}")
display(choice_df)

In [0]:
import mlflow
mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
with mlflow.start_run(run_name="E10_tune") as run:
    mlflow.set_tags({"module": "E10_tune", "git_commit": GIT_COMMIT})
    mlflow.log_params({"config_hash": CONFIG_HASH, "seed": SEED, "lightgbm_trials": TUNE["lightgbm_trials"],
                       "ebm_trials": TUNE["ebm_trials"], "C_trial": C_CHOSEN, "B_trial": B_CHOSEN,
                       "C_final_rounds": C_FINAL_ROUNDS, **{f"C_{k}": v for k, v in C_PARAMS.items()},
                       **{f"B_{k}": v for k, v in B_PARAMS.items()}})
    mlflow.log_metrics({"A_cv_auc": A_AUC_MEAN, "B_cv_auc": float(b_row.cv_auc_mean),
                        "C_cv_auc": float(c_row.cv_auc_mean), "B_t_auc": chosen["B"]["t_auc_mean"],
                        "C_t_auc": chosen["C"]["t_auc_mean"], "C_stage3_best_iter": S3_BEST_ITER})
    mlflow.log_artifact(XLSX); mlflow.log_artifact(CHOSEN_PATH)
    MLFLOW_RUN_ID = run.info.run_id
print(f"MLflow run: {MLFLOW_RUN_ID}")

In [0]:
# ── Acceptance (E10): re-read saved outputs and re-apply the stage-2 rules ──
print("=" * 60); print("ACCEPTANCE CRITERIA (E10 — tuning)"); print("=" * 60)
sx = pd.read_excel(XLSX, sheet_name=None)
with open(CHOSEN_PATH) as fh:
    cj = json.load(fh)

def reapply(model, names):
    trials = sx[f"{model}_trials"]
    sl, _ = common.shortlist_trials(trials, cfg, param_cols=names)
    tres = sx[f"{model}_tfold"]
    same_sl = sorted(sl.trial.tolist()) == sorted(tres.trial.tolist())
    ch, _ = common.pick_simplest(tres, model, cfg)
    row = trials.set_index("trial").loc[ch]
    p_saved = {k: cj[model]["params"][k] for k in names}
    same_p = all(np.isclose(float(row[k]), float(p_saved[k])) for k in names)
    return same_sl, ch, same_p

c_same_sl, c_re, c_same_p = reapply("C", C_PARAM_NAMES)
b_same_sl, b_re, b_same_p = reapply("B", B_PARAM_NAMES)
checks = [
    (f"{TUNE['lightgbm_trials']} C trials recorded (n={len(sx['C_trials'])})",
     len(sx["C_trials"]) == TUNE["lightgbm_trials"] and sx["C_trials"].cv_auc_mean.notna().all()),
    (f"{TUNE['ebm_trials']} B trials recorded (n={len(sx['B_trials'])})",
     len(sx["B_trials"]) == TUNE["ebm_trials"] and sx["B_trials"].cv_auc_mean.notna().all()),
    ("C: re-applied shortlist == saved T-fold set", c_same_sl),
    ("B: re-applied shortlist == saved T-fold set", b_same_sl),
    (f"C: re-applied selection -> trial {c_re} == chosen {cj['C']['trial']}, params identical",
     c_re == cj["C"]["trial"] and c_same_p),
    (f"B: re-applied selection -> trial {b_re} == chosen {cj['B']['trial']}, params identical",
     b_re == cj["B"]["trial"] and b_same_p),
    (f"Shortlists have <= {TUNE['max_candidates']} trials",
     len(sx["C_tfold"]) <= TUNE["max_candidates"] and len(sx["B_tfold"]) <= TUNE["max_candidates"]),
    (f"No OOT rows read (OOT-dated rows loaded = {N_OOT_ROWS_READ}; windows loaded = {sorted(master_dev['window'].unique())})",
     N_OOT_ROWS_READ == 0 and set(master_dev["window"].unique()) == {"DEV"}),
    ("All fold ids within DEV GOOD+BAD", set(rf.app_id.astype(str)) | set(tf.app_id.astype(str)) <= set(dev_gb.app_id)),
    (f"Model A 15-fold reference AUC computed (n={len(sx['A_cv'])})", len(sx["A_cv"]) == len(R_IDX)),
    (f"C final rounds = round(stage-3 best_iter x {TUNE['lgb_round_multiplier']}) = {cj['C']['final_rounds']}",
     cj["C"]["final_rounds"] == rnd(cj["C"]["stage3_best_iter"] * TUNE["lgb_round_multiplier"]) and cj["C"]["final_rounds"] > 0),
    ("C monotone vector aligned with columns", len(cj["C"]["monotone_constraints"]) == len(cj["C"]["columns"])),
    ("Report has meta sheet first", list(sx)[0] == "meta"),
]
all_pass = all(ok for _, ok in checks)
for desc, ok in checks:
    status = "\u2705 PASS" if ok else "\u274c FAIL"
    print(f"  {status} : {desc}")

summary = {"module": "E10_tune", "run_time": datetime.now().isoformat(), "elapsed_s": round(time.time() - _t0, 1),
           "git_commit": GIT_COMMIT, "config_hash": CONFIG_HASH, "rows_dev_good_bad": len(dev_gb),
           "bads_dev": int(y.sum()), "oot_rows_read": N_OOT_ROWS_READ,
           "A_cv_auc_mean": round(A_AUC_MEAN, 4), "B_trial": B_CHOSEN, "B_cv_auc_mean": round(float(b_row.cv_auc_mean), 4),
           "B_t_auc_mean": round(chosen["B"]["t_auc_mean"], 4), "C_trial": C_CHOSEN,
           "C_cv_auc_mean": round(float(c_row.cv_auc_mean), 4), "C_t_auc_mean": round(chosen["C"]["t_auc_mean"], 4),
           "C_final_rounds": C_FINAL_ROUNDS, "mlflow_run_id": MLFLOW_RUN_ID,
           "pass_fail": "PASS" if all_pass else "FAIL"}
with open(os.path.join(RUN_SUMS, "s04_tune.json"), "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print(f"\nRun summary: {RUN_SUMS}/s04_tune.json")
for desc, ok in checks:
    assert ok, f"ACCEPTANCE FAILED: {desc}"
print("\n\u2705 All E10 acceptance criteria passed.")